# Devoir maison n°2

In [ ]:

import os
import time
import warnings
 
import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import Perceptron
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

Récupération des données

In [4]:

def load_data():
    try:
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                            as_frame=False, parser="auto")
        return X.astype(np.float32) / 255.0, y.astype(int), "MNIST 784 px"
    except Exception as e:  # pas de réseau : repli sur digits 8x8 (test seulement)
        print("MNIST indisponible (", type(e).__name__, ") -> repli sur load_digits")
        from sklearn.datasets import load_digits
        d = load_digits()
        return d.data / 16.0, d.target, "digits 8x8 (REPLI, à ne pas rendre)"

OUT = "resultats"
os.makedirs(OUT, exist_ok=True)
SEED = 0
N_MAX = 200
STEPS = [10, 50, 100, 200]          # étapes de boosting évaluées
FAST = os.environ.get("FAST") == "1"  # mode test rapide
 

In [3]:
TASKS = {
    "3_vs_8": (lambda y: np.isin(y, [3, 8]), lambda y: (y == 8).astype(int)),
    "4_vs_9": (lambda y: np.isin(y, [4, 9]), lambda y: (y == 9).astype(int)),
    "0-4_vs_5-9": (lambda y: np.ones_like(y, bool), lambda y: (y >= 5).astype(int)),
}
 
WEAK = {
    "stump (prof.1)": lambda: DecisionTreeClassifier(max_depth=1, random_state=SEED),
    "arbre prof.2": lambda: DecisionTreeClassifier(max_depth=2, random_state=SEED),
    "arbre prof.3": lambda: DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "arbre prof.5": lambda: DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "perceptron": lambda: Perceptron(max_iter=5, tol=None, random_state=SEED),
}

In [5]:

def make_ada(base, n):
    # Compatibilité entre versions de scikit-learn :
    #  - récentes (>=1.8) : plus de paramètre `algorithm` (SAMME par défaut)
    #  - 1.2 à 1.7 : `estimator` + algorithm="SAMME" (nécessaire pour le perceptron)
    #  - anciennes : `base_estimator`
    for kw in (dict(estimator=base), dict(estimator=base, algorithm="SAMME"),
               dict(base_estimator=base, algorithm="SAMME")):
        try:
            return AdaBoostClassifier(n_estimators=n, random_state=SEED, **kw)
        except TypeError:
            continue
    raise RuntimeError("Version de scikit-learn non supportée")

In [6]:

def main():
    X, y, name = load_data()
    print("Données :", name, X.shape)
    n_max = 20 if FAST else N_MAX
    steps = [s for s in STEPS if s <= n_max] or [n_max]
    rows, cms = [], []
 
    for task, (mask_fn, label_fn) in TASKS.items():
        m = mask_fn(y)
        Xt, yt = X[m], label_fn(y[m])
        if not FAST and len(yt) > 20000:      # sous-échantillonnage pour 0-4 vs 5-9
            idx = np.random.RandomState(SEED).choice(len(yt), 20000, replace=False)
            Xt, yt = Xt[idx], yt[idx]
        Xtr, Xte, ytr, yte = train_test_split(
            Xt, yt, test_size=0.25, stratify=yt, random_state=SEED)
 
        fig, axes = plt.subplots(len(WEAK), len(steps),
                                 figsize=(2.6 * len(steps), 2.4 * len(WEAK)),
                                 squeeze=False)
        for i, (wname, mk) in enumerate(WEAK.items()):
            t0 = time.time()
            clf = make_ada(mk(), n_max).fit(Xtr, ytr)
            preds = {k: p for k, p in enumerate(clf.staged_predict(Xte), 1)}
            print(f"{task:12s} {wname:15s} fit {time.time()-t0:5.1f}s "
                  f"({len(clf.estimators_)} étapes)")
            for j, s in enumerate(steps):
                s_eff = min(s, len(clf.estimators_))
                p = preds[s_eff]
                cm = confusion_matrix(yte, p, labels=[0, 1])
                pr, rc, f1, _ = precision_recall_fscore_support(
                    yte, p, average="binary", zero_division=0)
                acc = (p == yte).mean()
                rows.append(dict(tache=task, faible=wname, etapes=s,
                                 accuracy=acc, precision=pr, rappel=rc, f1=f1))
                cms.append(dict(tache=task, faible=wname, etapes=s,
                                TN=cm[0, 0], FP=cm[0, 1], FN=cm[1, 0], TP=cm[1, 1]))
                ax = axes[i, j]
                ax.imshow(cm, cmap="Blues")
                for a in range(2):
                    for b in range(2):
                        ax.text(b, a, cm[a, b], ha="center", va="center",
                                color="white" if cm[a, b] > cm.max() / 2 else "black",
                                fontsize=9)
                ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
                ax.set_title(f"{wname}\n{s} étapes, acc={acc:.3f}", fontsize=7)
                ax.tick_params(labelsize=7)
                if i == len(WEAK) - 1:
                    ax.set_xlabel("prédit", fontsize=7)
                if j == 0:
                    ax.set_ylabel("réel", fontsize=7)
        fig.suptitle(f"Matrices de confusion — {task} (classe 1 = chiffre le plus grand / groupe 5-9)",
                     fontsize=9)
        fig.tight_layout()
        fig.savefig(f"{OUT}/matrices_confusion_{task}.png", dpi=130)
        plt.close(fig)
 
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT}/resume.csv", index=False)
    pd.DataFrame(cms).to_csv(f"{OUT}/matrices_confusion.csv", index=False)
    print("\n=== Accuracy (test) ===")
    print(df.pivot_table(index=["tache", "faible"], columns="etapes",
                         values="accuracy").round(3).to_string())
 
 
if __name__ == "__main__":
    main()

Données : MNIST 784 px (70000, 784)
3_vs_8       stump (prof.1)  fit  41.2s (200 étapes)
3_vs_8       arbre prof.2    fit  75.7s (200 étapes)
3_vs_8       arbre prof.3    fit 110.6s (200 étapes)
3_vs_8       arbre prof.5    fit 176.6s (200 étapes)
3_vs_8       perceptron      fit   0.5s (5 étapes)
4_vs_9       stump (prof.1)  fit  36.0s (200 étapes)
4_vs_9       arbre prof.2    fit  61.7s (200 étapes)
4_vs_9       arbre prof.3    fit  92.5s (200 étapes)
4_vs_9       arbre prof.5    fit 138.6s (200 étapes)
4_vs_9       perceptron      fit   1.3s (13 étapes)
0-4_vs_5-9   stump (prof.1)  fit  56.3s (200 étapes)
0-4_vs_5-9   arbre prof.2    fit 102.0s (200 étapes)
0-4_vs_5-9   arbre prof.3    fit 147.6s (200 étapes)
0-4_vs_5-9   arbre prof.5    fit 244.5s (200 étapes)
0-4_vs_5-9   perceptron      fit   3.5s (23 étapes)

=== Accuracy (test) ===
etapes                       10     50     100    200
tache      faible                                    
0-4_vs_5-9 arbre prof.2    0.830  0.860 